# 🤖 LangChain Agents — A Complete Teaching Notebook

---

## What is an Agent?

In LangChain, an **Agent** is an LLM that doesn't just answer a question — it **decides what actions to take** to answer it.

Instead of you wiring up logic like:
> *"If the user asks about weather, call weather API. If they ask about math, use a calculator."*

…you hand the LLM a set of **tools** and let it **reason** about which tools to call, in what order, and when to stop.

### The Core Loop (ReAct Pattern)

```
User Input
    ↓
LLM THINKS  →  "I need to search for X"
    ↓
LLM ACTS    →  calls search tool("X")
    ↓
OBSERVATION →  "Result: ..."
    ↓
LLM THINKS  →  "Now I have enough info"
    ↓
FINAL ANSWER
```

This is called the **ReAct** (Reason + Act) loop.

---

## 📚 What This Notebook Covers

| Section | Topic |
|---------|-------|
| 1 | Setup & Installation |
| 2 | Tools — the building blocks of agents |
| 3 | Your First Agent (ReAct) |
| 4 | Agent with Multiple Tools |
| 5 | Agent Memory |
| 6 | Custom Tools with `@tool` decorator |
| 7 | Structured Tool Output |
| 8 | OpenAI Functions Agent |
| 9 | Streaming Agent Output |
| 10 | 🚀 Mini Project: Research Assistant Agent |

---
## Section 1 — Setup & Installation

Install the required packages. We use:
- `langchain` — core framework
- `langchain-openai` — OpenAI integration
- `langchain-community` — community tools (Wikipedia, etc.)
- `duckduckgo-search` — free web search tool
- `wikipedia` — Wikipedia API wrapper

In [ ]:
!pip install -qU langchain langchain-openai langchain-community duckduckgo-search wikipedia

In [ ]:
import os
from getpass import getpass

# Set your OpenAI API key
os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")

print("✅ API Key set!")

---
## Section 2 — Tools: The Building Blocks of Agents

**Tools** are functions that agents can call. Every tool has:
- A **name** — how the agent refers to it
- A **description** — what the LLM reads to decide *when* to use it
- A **function** — what actually runs

Think of tools as the agent's "hands" — without them, the agent can only think, not act.

LangChain provides many built-in tools. Let's explore a few.

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.tools import Tool

# --- Tool 1: Web Search ---
search_tool = DuckDuckGoSearchRun()

# --- Tool 2: Wikipedia ---
wiki_api = WikipediaAPIWrapper(top_k_results=2, doc_content_chars_max=500)
wiki_tool = WikipediaQueryRun(api_wrapper=wiki_api)

# --- Tool 3: Python Calculator (using eval safely) ---
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression. Input must be a valid Python math expression."""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Error: {e}"

calculator_tool = Tool(
    name="Calculator",
    func=calculate,
    description="Useful for evaluating math expressions. Input: a Python math expression like '2**10' or '(15 * 3) / 2'."
)

# Test tools directly
print("🔍 Search result snippet:")
print(search_tool.run("current Python version 2024")[:200])

print("\n📖 Wikipedia snippet:")
print(wiki_tool.run("LangChain framework")[:200])

print("\n🔢 Calculator:")
print(calculator_tool.run("2 ** 10"))

### Tool Anatomy

You can inspect a tool's metadata — this is exactly what the LLM sees when deciding which tool to use:

In [ ]:
# Inspect tool metadata — this is what the LLM "reads"
for tool in [search_tool, wiki_tool, calculator_tool]:
    print(f"Name        : {tool.name}")
    print(f"Description : {tool.description}")
    print("-" * 60)

---
## Section 3 — Your First Agent (ReAct)

The **ReAct** agent is the classic agent pattern. It works by prompting the LLM to output:
- `Thought:` — what it's thinking
- `Action:` — which tool to call
- `Action Input:` — what to pass to the tool
- `Observation:` — result from the tool
- ...repeat until...
- `Final Answer:` — the response to the user

### Modern LangChain uses LCEL (LangChain Expression Language)

With `create_react_agent` + `AgentExecutor`, you get a clean, production-ready setup.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain import hub
from langchain.agents import create_react_agent, AgentExecutor

# 1. The LLM brain
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2. Tools the agent can use
tools = [search_tool, calculator_tool]

# 3. The ReAct prompt — pulled from LangChain Hub
#    This is a carefully crafted prompt that enables the Thought/Action/Observation loop
react_prompt = hub.pull("hwchase17/react")

print("ReAct Prompt Template:")
print("=" * 60)
print(react_prompt.template[:800])
print("...")

In [ ]:
# 4. Create the agent (just the reasoning logic)
react_agent = create_react_agent(llm=llm, tools=tools, prompt=react_prompt)

# 5. Wrap it in AgentExecutor (handles the loop, errors, max iterations)
agent_executor = AgentExecutor(
    agent=react_agent,
    tools=tools,
    verbose=True,          # 👀 Show the Thought/Action/Observation loop
    max_iterations=5,      # Safety limit
    handle_parsing_errors=True
)

# 6. Run it!
result = agent_executor.invoke({
    "input": "What is 25 to the power of 3? Also search for what LangChain is used for."
})

print("\n" + "=" * 60)
print("FINAL ANSWER:", result["output"])

---
## Section 4 — Agent with Multiple Tools

The real power of agents is combining tools intelligently. Let's give our agent **three tools** and ask it a question that requires using more than one.

In [ ]:
# Rebuild agent with all 3 tools
all_tools = [search_tool, wiki_tool, calculator_tool]

multi_tool_agent = create_react_agent(llm=llm, tools=all_tools, prompt=react_prompt)

multi_executor = AgentExecutor(
    agent=multi_tool_agent,
    tools=all_tools,
    verbose=True,
    max_iterations=7,
    handle_parsing_errors=True
)

# This question needs BOTH Wikipedia AND Calculator
response = multi_executor.invoke({
    "input": "According to Wikipedia, in what year was Python programming language created? "
             "Then calculate how many years ago that was from 2025."
})

print("\nFINAL ANSWER:", response["output"])

---
## Section 5 — Agent Memory

By default, agents are **stateless** — they forget previous messages. This is a problem for multi-turn conversations.

We fix this with **memory**. LangChain provides `ConversationBufferMemory` which stores the full conversation history and injects it into every call.

### Two Types of Memory to Know

| Memory Type | What it stores | Best for |
|---|---|---|
| `ConversationBufferMemory` | Full history | Short conversations |
| `ConversationSummaryMemory` | Summarized history | Long conversations |

In [ ]:
from langchain.memory import ConversationBufferMemory
from langchain import hub
from langchain.agents import create_react_agent, AgentExecutor

# Pull the memory-aware ReAct prompt
react_chat_prompt = hub.pull("hwchase17/react-chat")

# Create memory — IMPORTANT: memory_key must match the prompt variable name
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

# Create agent with memory
memory_agent = create_react_agent(
    llm=llm,
    tools=all_tools,
    prompt=react_chat_prompt
)

memory_executor = AgentExecutor(
    agent=memory_agent,
    tools=all_tools,
    memory=memory,
    verbose=True,
    max_iterations=5,
    handle_parsing_errors=True
)

# Turn 1
print("--- Turn 1 ---")
r1 = memory_executor.invoke({"input": "My name is Alice and I love Python programming."})
print("Agent:", r1["output"])

# Turn 2 — tests if agent remembers
print("\n--- Turn 2 ---")
r2 = memory_executor.invoke({"input": "What programming language did I say I love?"})
print("Agent:", r2["output"])

In [ ]:
# Peek at what's stored in memory
print("Memory contents:")
for msg in memory.chat_memory.messages:
    role = "Human" if msg.type == "human" else "AI"
    print(f"  [{role}]: {msg.content[:100]}")

---
## Section 6 — Custom Tools with the `@tool` Decorator

You can turn **any Python function** into an agent tool using the `@tool` decorator.

Key rules for good tools:
1. The **docstring** becomes the tool description — write it clearly for the LLM
2. Use **type hints** — LangChain uses them to validate inputs
3. Return a **string** — agents expect string observations

In [ ]:
from langchain.tools import tool
from datetime import datetime
import json

# Custom Tool 1: Current date/time
@tool
def get_current_datetime(format: str = "%Y-%m-%d %H:%M") -> str:
    """Returns the current date and time. Useful when the user asks about today's date,
    current time, or what day it is. Input: optional datetime format string."""
    return datetime.now().strftime(format)


# Custom Tool 2: Word counter
@tool
def count_words(text: str) -> str:
    """Counts the number of words and characters in a given text. 
    Useful when the user wants to know the length of a text.
    Input: the text string to analyze."""
    words = len(text.split())
    chars = len(text)
    return json.dumps({"word_count": words, "character_count": chars})


# Custom Tool 3: Unit converter
@tool
def convert_units(query: str) -> str:
    """Converts between common units. Supports: km to miles, celsius to fahrenheit,
    kg to pounds. Input format: '<value> <from_unit> to <to_unit>', e.g. '100 km to miles'."""
    try:
        parts = query.lower().replace(" to ", "|").split("|")
        value_and_unit = parts[0].strip().split()
        value = float(value_and_unit[0])
        from_unit = value_and_unit[1]
        to_unit = parts[1].strip()
        
        conversions = {
            ("km", "miles"): lambda v: v * 0.621371,
            ("miles", "km"): lambda v: v * 1.60934,
            ("celsius", "fahrenheit"): lambda v: v * 9/5 + 32,
            ("fahrenheit", "celsius"): lambda v: (v - 32) * 5/9,
            ("kg", "pounds"): lambda v: v * 2.20462,
            ("pounds", "kg"): lambda v: v * 0.453592,
        }
        
        key = (from_unit, to_unit)
        if key in conversions:
            result = conversions[key](value)
            return f"{value} {from_unit} = {result:.4f} {to_unit}"
        return f"Conversion from {from_unit} to {to_unit} not supported."
    except Exception as e:
        return f"Error parsing query: {e}. Use format like '100 km to miles'"


# Test each tool directly
print(get_current_datetime.invoke("%A, %B %d, %Y"))
print(count_words.invoke("The quick brown fox jumps over the lazy dog"))
print(convert_units.invoke("100 km to miles"))

In [ ]:
# Build agent with all custom tools
custom_tools = [get_current_datetime, count_words, convert_units, calculator_tool]

custom_agent = create_react_agent(llm=llm, tools=custom_tools, prompt=react_prompt)
custom_executor = AgentExecutor(
    agent=custom_agent,
    tools=custom_tools,
    verbose=True,
    max_iterations=5,
    handle_parsing_errors=True
)

result = custom_executor.invoke({
    "input": "What is today's date? Also, convert 42 celsius to fahrenheit."
})
print("\nFINAL ANSWER:", result["output"])

---
## Section 7 — Structured Tool Output with `StructuredTool`

When a tool needs **multiple input parameters** (not just a single string), use `StructuredTool` with a Pydantic schema.

This is important for real-world tools like API calls, database queries, etc.

In [ ]:
from langchain.tools import StructuredTool
from pydantic import BaseModel, Field

# Define the input schema using Pydantic
class BMIInput(BaseModel):
    weight_kg: float = Field(description="Weight in kilograms")
    height_m: float = Field(description="Height in meters")

# The actual function
def calculate_bmi(weight_kg: float, height_m: float) -> str:
    """Calculate Body Mass Index (BMI) from weight and height."""
    bmi = weight_kg / (height_m ** 2)
    if bmi < 18.5:
        category = "Underweight"
    elif bmi < 25:
        category = "Normal weight"
    elif bmi < 30:
        category = "Overweight"
    else:
        category = "Obese"
    return f"BMI: {bmi:.1f} ({category})"

# Wrap as a StructuredTool
bmi_tool = StructuredTool.from_function(
    func=calculate_bmi,
    name="BMI_Calculator",
    description="Calculates Body Mass Index given weight in kg and height in meters.",
    args_schema=BMIInput
)

# Test
print(bmi_tool.run({"weight_kg": 70, "height_m": 1.75}))

In [ ]:
# Use structured tool in an agent
structured_tools = [bmi_tool, calculator_tool]

structured_agent = create_react_agent(llm=llm, tools=structured_tools, prompt=react_prompt)
structured_executor = AgentExecutor(
    agent=structured_agent,
    tools=structured_tools,
    verbose=True,
    max_iterations=4,
    handle_parsing_errors=True
)

result = structured_executor.invoke({
    "input": "What is the BMI of someone who weighs 80 kg and is 1.80 meters tall? Is that healthy?"
})
print("\nFINAL ANSWER:", result["output"])

---
## Section 8 — OpenAI Functions Agent (More Reliable)

The **OpenAI Functions Agent** is more reliable than the classic ReAct agent for OpenAI models.

Instead of the LLM generating text like `Action: search` (which can have formatting errors), it uses OpenAI's **function calling API** — a structured JSON protocol that is:
- ✅ More reliable (no parsing errors)
- ✅ Handles multi-argument tools natively
- ✅ Better at knowing when NOT to call a tool

This is the **recommended approach** when using OpenAI models.

In [ ]:
from langchain.agents import create_openai_functions_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# The prompt must include an 'agent_scratchpad' MessagesPlaceholder
openai_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the available tools to answer questions accurately."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

tools_for_openai = [search_tool, wiki_tool, calculator_tool, get_current_datetime]

# Create OpenAI Functions Agent
openai_agent = create_openai_functions_agent(
    llm=llm,
    tools=tools_for_openai,
    prompt=openai_prompt
)

openai_executor = AgentExecutor(
    agent=openai_agent,
    tools=tools_for_openai,
    verbose=True,
    max_iterations=5
)

result = openai_executor.invoke({
    "input": "What is today's date and what is 144 divided by 12?"
})
print("\nFINAL ANSWER:", result["output"])

---
## Section 9 — Streaming Agent Output

For better UX, you can **stream** agent output — printing tokens as they arrive instead of waiting for the full response.

This is important for production apps where users see the agent "thinking" in real-time.

In [ ]:
# Stream agent events
print("Streaming agent output:\n")
print("=" * 60)

for event in openai_executor.stream({"input": "What is 2 to the power of 8? Show your work."}):
    # Events can be: 'actions', 'steps', 'output'
    if "actions" in event:
        for action in event["actions"]:
            print(f"🔧 Tool call  : {action.tool}")
            print(f"   Input     : {action.tool_input}")
    
    elif "steps" in event:
        for step in event["steps"]:
            print(f"📥 Observation: {step.observation}")
    
    elif "output" in event:
        print(f"\n✅ Final Answer: {event['output']}")

---
## Section 10 — Agent Callbacks (Observability)

**Callbacks** let you hook into every step of the agent's execution — great for logging, monitoring, and debugging in production.

In [ ]:
from langchain.callbacks.base import BaseCallbackHandler
from typing import Any, Dict, List, Union

class AgentLogger(BaseCallbackHandler):
    """Custom callback to log agent actions cleanly."""
    
    def on_agent_action(self, action, **kwargs):
        print(f"\n🤔 THINKING → Using tool: [{action.tool}]")
        print(f"   Input: {action.tool_input}")
    
    def on_tool_end(self, output: str, **kwargs):
        print(f"   Result: {output[:150]}..." if len(output) > 150 else f"   Result: {output}")
    
    def on_agent_finish(self, finish, **kwargs):
        print(f"\n🏁 FINISHED")

# Create executor with callback
logged_executor = AgentExecutor(
    agent=openai_agent,
    tools=tools_for_openai,
    callbacks=[AgentLogger()],
    verbose=False,   # Turn off default verbose since we have custom logging
    max_iterations=4
)

result = logged_executor.invoke({
    "input": "Search for what GPT-4 is and tell me 256 * 16."
})
print(f"\n💬 Final: {result['output']}")

---
# 🚀 Mini Project: Personal Research Assistant Agent

## Project Overview

We'll build a **Research Assistant Agent** that can:
1. 🔍 Search the web for current information
2. 📖 Look up detailed Wikipedia entries
3. 🧮 Perform calculations
4. 📅 Track and report the current date
5. 📝 Save notes to a research log (persists in memory)
6. 📊 Summarize findings in a structured report

The agent maintains **conversation memory** so you can have a multi-turn research session.

---

In [ ]:
# ============================================================
# MINI PROJECT: Research Assistant Agent
# ============================================================

from langchain_openai import ChatOpenAI
from langchain.agents import create_openai_functions_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.memory import ConversationBufferMemory
from langchain.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from datetime import datetime
import json

# ---- Research Notes Store ----
research_notes = []

# ---- Custom Tools ----

@tool
def save_research_note(note: str) -> str:
    """Save an important finding or note to the research log.
    Use this to remember key facts discovered during research.
    Input: the note or finding to save as a string."""
    timestamp = datetime.now().strftime("%H:%M:%S")
    entry = {"time": timestamp, "note": note}
    research_notes.append(entry)
    return f"✅ Note saved at {timestamp}: '{note[:80]}...'"


@tool
def view_research_notes(query: str = "") -> str:
    """View all saved research notes from this session.
    Use this to review what has been found so far.
    Input: optional keyword to filter notes (or empty string to see all)."""
    if not research_notes:
        return "No research notes saved yet."
    
    filtered = [
        f"[{n['time']}] {n['note']}"
        for n in research_notes
        if not query or query.lower() in n['note'].lower()
    ]
    return "Research Notes:\n" + "\n".join(filtered) if filtered else "No matching notes."


@tool
def get_current_date(format: str = "%B %d, %Y") -> str:
    """Returns the current date. Use when the user asks about today's date
    or when you need to add timestamps to research.
    Input: optional date format string (default: 'Month DD, YYYY')."""
    return datetime.now().strftime(format)


@tool  
def calculate(expression: str) -> str:
    """Evaluates a mathematical expression. Use for any numerical calculations.
    Input: a valid Python math expression like '(15 * 8) / 3' or '2 ** 10'."""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return f"{expression} = {result}"
    except Exception as e:
        return f"Calculation error: {e}"


# ---- Built-in tools ----
search = DuckDuckGoSearchRun()
wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(top_k_results=2, doc_content_chars_max=800))

# ---- All project tools ----
project_tools = [
    search,
    wiki,
    calculate,
    get_current_date,
    save_research_note,
    view_research_notes,
]

print("✅ Tools ready:")
for t in project_tools:
    print(f"  - {t.name}")

In [ ]:
# ---- System Prompt ----
SYSTEM_PROMPT = """
You are a thorough and organized Research Assistant.

Your job is to help the user research topics by:
1. Searching for current information using the search tool
2. Looking up detailed background using Wikipedia
3. Saving important findings using save_research_note
4. Performing any required calculations
5. Providing clear, well-structured summaries

Guidelines:
- Always save key findings as notes so they're not lost
- Be specific and cite your sources when possible
- If asked to summarize findings, call view_research_notes first
- Today's date can be found with the get_current_date tool

You maintain memory across the conversation, so you can build on previous research.
"""

# ---- Prompt Template ----
research_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

# ---- LLM ----
research_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)

# ---- Memory ----
research_memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

# ---- Build Agent ----
research_agent = create_openai_functions_agent(
    llm=research_llm,
    tools=project_tools,
    prompt=research_prompt
)

research_executor = AgentExecutor(
    agent=research_agent,
    tools=project_tools,
    memory=research_memory,
    verbose=True,
    max_iterations=8,
    handle_parsing_errors=True
)

print("🤖 Research Assistant Agent is ready!")

In [ ]:
# ============================================================
# RESEARCH SESSION: Exploring Artificial Intelligence
# ============================================================

def chat(user_input: str):
    """Helper function to chat with the research assistant."""
    print(f"\n{'='*65}")
    print(f"👤 You: {user_input}")
    print("="*65)
    result = research_executor.invoke({"input": user_input})
    print(f"\n🤖 Assistant: {result['output']}")
    return result["output"]

# --- Research Turn 1: Kick off the research ---
chat("I want to research the history of artificial intelligence. "
     "Search for when AI was founded and save the key dates as notes.")

In [ ]:
# --- Research Turn 2: Dig deeper with Wikipedia ---
chat("Look up the Turing Test on Wikipedia and save what year it was proposed.")

In [ ]:
# --- Research Turn 3: Use the calculator ---
chat("The Turing Test was proposed in 1950. Calculate how many years ago that was from 2025. "
     "Also search for the latest GPT model from OpenAI and save that too.")

In [ ]:
# --- Research Turn 4: Ask for a summary using saved notes ---
chat("Great research session! Please look at all my saved notes and write me "
     "a structured summary report of what we learned about AI today.")

In [ ]:
# ---- View all saved research notes ----
print("\n" + "="*65)
print("📋 ALL SAVED RESEARCH NOTES")
print("="*65)
for i, note in enumerate(research_notes, 1):
    print(f"  [{i}] {note['time']} — {note['note']}")

---
## 🎯 Try It Yourself!

The research assistant is still running with memory. Try these queries:

In [ ]:
# Try your own research questions!
# Examples to try:
# chat("Research the history of Python programming language and save key facts.")
# chat("Who founded OpenAI? Search for it and save the founders' names.")
# chat("What is quantum computing? Look it up on Wikipedia.")
# chat("What did we research in this session? Give me a summary.")

chat("What's today's date? Also, do a quick search on recent developments in large language models.")

---
## 📚 Key Takeaways

| Concept | What you learned |
|---|---|
| **ReAct Agent** | LLM loops through Thought → Action → Observation |
| **AgentExecutor** | Manages the loop, limits, error handling |
| **Tools** | Functions the agent can call to interact with the world |
| **`@tool` decorator** | Turn any Python function into an agent tool |
| **StructuredTool** | Multi-argument tools with Pydantic validation |
| **Memory** | `ConversationBufferMemory` gives agents state across turns |
| **OpenAI Functions Agent** | More reliable tool calling via structured JSON |
| **Callbacks** | Hook into agent lifecycle for logging/monitoring |
| **Streaming** | Stream events for real-time UX |

## 🔮 What's Next?

- **LangGraph** — for complex multi-agent workflows with branching logic
- **LangSmith** — observability and tracing for production agents  
- **Tool Retrieval** — dynamically select tools from a large toolset using embeddings
- **Multi-Agent Systems** — agents that orchestrate other agents